CNN + Shifter on **Sensorium** dataset (Table 1 style)

Same model as train_cnn_shifter_table_1 but data from Sensorium layout:
- `data/videos/{trial}.npy`, `data/responses/{trial}.npy`, `data/behavior/{trial}.npy`, `data/pupil_center/{trial}.npy`
- `meta/trials/tiers.npy` for train/validation/test split
- Responses standardized as r/std_r (competition requirement)

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Subset, DataLoader, ConcatDataset
from mouse_model.sensorium_dataset import SensoriumDataset
import numpy as np
from mouse_model.evaluation import cor_in_time
from sklearn.metrics import r2_score, mean_squared_error
import random, os
from kornia.geometry.transform import get_affine_matrix2d, warp_affine

In [2]:
class Shifter(nn.Module):
    def __init__(self, input_dim=4, output_dim=3, hidden_dim=256):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        self.layers = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim),
            nn.Tanh(),
        )
        self.bias = nn.Parameter(torch.zeros(3))
    def forward(self, x):
        x = x.reshape(-1,self.input_dim )
        x = self.layers(x)
        x0 = (x[...,0] + self.bias[0]) * 80/4
        x1 = (x[...,1] + self.bias[1]) * 60/4
        x2 = (x[...,2] + self.bias[2]) * 180/4
        x = torch.stack([x0, x1, x2], dim=-1)
        x = x.reshape(-1,1,self.output_dim)
        return x

In [3]:
# useful for printing in nn.Sequential
class PrintLayer(nn.Module):
    
    def __init__(self):
        super(PrintLayer, self).__init__()
    
    def forward(self, x):
        print(x.shape)
        return x

def size_helper(in_length, kernel_size, padding=0, dilation=1, stride=1):
    # https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html#torch.nn.Conv2d
    res = in_length + 2 * padding - dilation * (kernel_size - 1) - 1
    res /= stride
    res += 1
    return np.floor(res)

# CNN, the last fully connected layer maps to output_dim
class VisualEncoder(nn.Module):
    
    def __init__(self, output_dim, input_shape=(60, 80), k1=7, k2=7, k3=7):
        
        super().__init__()
        
        self.input_shape = (60, 80)
        out_shape_0 = size_helper(in_length=input_shape[0], kernel_size=k1, stride=2)
        out_shape_0 = size_helper(in_length=out_shape_0, kernel_size=k2, stride=2)
        out_shape_0 = size_helper(in_length=out_shape_0, kernel_size=k3, stride=2)
        out_shape_1 = size_helper(in_length=input_shape[1], kernel_size=k1, stride=2)
        out_shape_1 = size_helper(in_length=out_shape_1, kernel_size=k2, stride=2)
        out_shape_1 = size_helper(in_length=out_shape_1, kernel_size=k3, stride=2)
        self.output_shape = (int(out_shape_0), int(out_shape_1)) # shape of the final feature map
        
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=128, kernel_size=k1, stride=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Conv2d(in_channels=128, out_channels=64, kernel_size=k2, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Conv2d(in_channels=64, out_channels=32, kernel_size=k3, stride=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Flatten(),
            nn.Linear(self.output_shape[0]*self.output_shape[1]*32, output_dim)
        )
        
    def forward(self, x):

        x = self.layers(x)

        return x

In [4]:
class Predictor(nn.Module):
    
    def __init__(self, num_neurons):

        super().__init__()
        
        self.encoder = VisualEncoder(output_dim=num_neurons)
        self.softplus = nn.Softplus()
        self.shifter = Shifter()

    def forward(self, images, behav):
        # print(images.shape)  torch.Size([256, 1, 60, 80])
        if args.shifter:
            bs = images.size()[0]
            behav_shifter = torch.concat((behav[...,4].unsqueeze(-1),   # theta
                                          behav[...,3].unsqueeze(-1),   # phi
                                          behav[...,1].unsqueeze(-1),  # pitch
                                         behav[...,2].unsqueeze(-1),  # roll
                                         ), dim=-1)  
            shift_param = self.shifter(behav_shifter)  
            shift_param = shift_param.reshape(-1,3)
            scale_param = torch.ones_like(shift_param[..., 0:2]).to(shift_param.device)
            affine_mat = get_affine_matrix2d(
                                            translations=shift_param[..., 0:2] ,
                                             scale = scale_param, 
                                             center =torch.repeat_interleave(torch.tensor([[30,40]], dtype=torch.float), 
                                                                            bs*1, dim=0).to(shift_param.device), 
                                             angle=shift_param[..., 2])
            affine_mat = affine_mat[:, :2, :]
            images = warp_affine(images, affine_mat, dsize=(60,80))
        pred = self.encoder(images)
        pred = self.softplus(pred)
        
        return pred

In [5]:
# Sensorium dataset path (dynamic Video dataset)
SENSORIUM_ROOT = "/home/herbelinluke/Downloads/dynamic29515-10-12-Video-9b4f6a1a067fe51e15306b9628efea20"

class Args:
    seed = 0
    sensorium_root = SENSORIUM_ROOT
    epochs = 100
    batch_size = 256
    seq_len = 1
    num_neurons = None  # set from dataset
    learning_rate = 0.0001
    best_train_path = None
    best_val_path = None
    shifter = False

    # --- Data split config ---
    # "tiers"  = only use the 'train' tier (348 trials)
    # "random" = use ALL tiers (711 trials), random split into train/val
    split_strategy = "random"
    train_ratio = 0.7             # fraction for training (rest → validation)
    max_train_samples = None       # None = use all, int = cap training set size

    # --- Frame mode ---
    # "mean"      = one sample per trial (mean frame → mean response)
    # "per_frame" = one sample per video frame (frame → binned response, like mouse dataset)
    vid_frame = "per_frame"

    # --- Neuron filtering config ---
    # Min per-neuron trial-to-trial std (standardized) to count as "high signal".
    # Neurons below this threshold are "easy wins" with little stimulus-driven variability.
    # Set to 0.0 to disable (keep all valid neurons).
    min_neuron_std = 0.1

args = Args()

seed = args.seed
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.cuda.empty_cache()
print(torch.cuda.is_available())
print(f"split_strategy={args.split_strategy}, train_ratio={args.train_ratio}, "
      f"max_train_samples={args.max_train_samples}, vid_frame={args.vid_frame}, "
      f"min_neuron_std={args.min_neuron_std}")

True
split_strategy=random, train_ratio=0.7, max_train_samples=None, vid_frame=per_frame, min_neuron_std=0.1


In [6]:
import pickle

def _get_or_create_split(full_ds, split_tag, train_ratio, seed):
    """Load a cached random split or create + save one."""
    ratio_str = str(int(train_ratio * 100))
    split_path = os.path.join(
        args.sensorium_root, "meta", "trials",
        f"split_{ratio_str}_{100 - int(train_ratio * 100)}_{split_tag}.pkl",
    )
    n = len(full_ds)
    if os.path.isfile(split_path):
        with open(split_path, "rb") as f:
            sp = pickle.load(f)
        if len(sp["train_indices"]) + len(sp["val_indices"]) == n:
            print(f"Loaded split from {split_path}")
            return sp["train_indices"], sp["val_indices"]

    n_train = int(n * train_ratio)
    indices = np.random.RandomState(seed).permutation(n)
    train_indices = indices[:n_train].tolist()
    val_indices = indices[n_train:].tolist()
    os.makedirs(os.path.dirname(split_path), exist_ok=True)
    with open(split_path, "wb") as f:
        pickle.dump({"train_indices": train_indices, "val_indices": val_indices,
                      "seed": seed, "train_ratio": train_ratio, "tag": split_tag}, f)
    print(f"Saved split to {split_path}")
    return train_indices, val_indices


def load_train_val_ds():
    if args.split_strategy == "tiers":
        # Train on the full 'train' tier; validate on all other tiers
        train_ds = SensoriumDataset(
            root_dir=args.sensorium_root, data_split="train",
            seq_len=args.seq_len, vid_frame=args.vid_frame, standardize_responses=True,
        )
        val_ds = SensoriumDataset(
            root_dir=args.sensorium_root, data_split="non_train",
            seq_len=args.seq_len, vid_frame=args.vid_frame, standardize_responses=True,
        )
        if args.max_train_samples is not None and len(train_ds) > args.max_train_samples:
            indices = np.random.RandomState(args.seed).permutation(len(train_ds))
            train_ds = Subset(train_ds, indices[: args.max_train_samples].tolist())
        if args.num_neurons is None:
            args.num_neurons = val_ds.num_neurons or train_ds.num_neurons
    elif args.split_strategy == "random":
        # Pool ALL tiers and do a random train_ratio split
        full_ds = SensoriumDataset(
            root_dir=args.sensorium_root, data_split="all",
            seq_len=args.seq_len, vid_frame=args.vid_frame, standardize_responses=True,
        )
        train_indices, val_indices = _get_or_create_split(
            full_ds, "all", args.train_ratio, args.seed,
        )
        if args.max_train_samples is not None and len(train_indices) > args.max_train_samples:
            train_indices = train_indices[: args.max_train_samples]
        train_ds = Subset(full_ds, train_indices)
        val_ds = Subset(full_ds, val_indices)
        if args.num_neurons is None:
            args.num_neurons = full_ds.num_neurons
    else:
        raise ValueError(f"Unknown split_strategy: {args.split_strategy}")

    print(f"split_strategy={args.split_strategy} | "
          f"train={len(train_ds)} val={len(val_ds)} | num_neurons={args.num_neurons}")
    return train_ds, val_ds

In [7]:
def load_test_ds():
    """Load a test set disjoint from training data.
    
    - tiers: all non-train tiers (same pool used for val during training).
    - random: the val holdout from the random split.
    """
    if args.split_strategy == "tiers":
        test_ds = SensoriumDataset(
            root_dir=args.sensorium_root, data_split="non_train",
            seq_len=args.seq_len, vid_frame=args.vid_frame, standardize_responses=True,
        )
        print(f"Test set: non-train tiers ({len(test_ds)} samples)")
        return test_ds
    else:
        full_ds = SensoriumDataset(
            root_dir=args.sensorium_root, data_split="all",
            seq_len=args.seq_len, vid_frame=args.vid_frame, standardize_responses=True,
        )
        _, val_indices = _get_or_create_split(full_ds, "all", args.train_ratio, args.seed)
        test_ds = Subset(full_ds, val_indices)
        print(f"Test set: random val holdout ({len(test_ds)} trials)")
        return test_ds

In [8]:
def _compute_neuron_masks(pred_val, label_val):
    """Return (valid, high_signal) boolean masks over neurons."""
    eps = 1e-12
    var_pred = np.var(pred_val, axis=0)
    var_label = np.var(label_val, axis=0)
    valid = (var_pred > eps) & (var_label > eps)
    # high_signal: valid AND per-neuron trial-to-trial std above threshold
    std_label = np.std(label_val, axis=0)
    high_signal = valid & (std_label >= args.min_neuron_std) if args.min_neuron_std > 0 else valid.copy()
    return valid, high_signal


def _metrics_for_mask(pred_val, label_val, mask, num_neurons):
    """Compute per-neuron cor/R²/EV and MSE for a given neuron mask."""
    cor_array = cor_in_time(pred_val, label_val)
    cor_pn = np.array(cor_array.flatten(), dtype=np.float64)
    cor_pn[~mask] = np.nan

    r2_pn = np.full(num_neurons, np.nan, dtype=np.float64)
    for j in np.where(mask)[0]:
        r2_pn[j] = r2_score(label_val[:, j], pred_val[:, j])

    eps = 1e-12
    var_label = np.var(label_val, axis=0)
    var_res = np.var(label_val - pred_val, axis=0)
    with np.errstate(divide="ignore", invalid="ignore"):
        ev_pn = np.where(var_label > eps, 1.0 - var_res / var_label, np.nan).astype(np.float64)
    ev_pn[~mask] = np.nan

    n = int(mask.sum())
    mse = mean_squared_error(label_val[:, mask], pred_val[:, mask]) if n > 0 else float("nan")
    return {
        "cor_pn": cor_pn, "mean_cor": float(np.nanmean(cor_pn)) if n > 0 else 0.0,
        "r2_pn": r2_pn, "mean_r2": float(np.nanmean(r2_pn)) if n > 0 else 0.0,
        "ev_pn": ev_pn, "mean_ev": float(np.nanmean(ev_pn)) if n > 0 else 0.0,
        "mse": mse, "n": n,
    }


def train_model():

    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(args.seed)

    train_ds, val_ds = load_train_val_ds()

    train_dataloader = DataLoader(dataset=train_ds, batch_size=args.batch_size, shuffle=True, num_workers=8)
    val_dataloader = DataLoader(dataset=val_ds, batch_size=args.batch_size, shuffle=False, num_workers=8)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate)

    best_train_loss = np.inf
    best_val_loss = np.inf
    train_loss_list = []
    val_loss_list = []
    # "all valid" metrics
    val_cor_list = []
    val_r2_list = []
    val_mse_list = []
    val_poisson_loss_list = []
    val_bits_per_spike_list = []
    val_explained_var_list = []
    # "high signal only" metrics
    hs_cor_list = []
    hs_r2_list = []
    hs_mse_list = []
    hs_explained_var_list = []
    # per-neuron history
    cor_per_neuron_per_epoch = []
    r2_per_neuron_per_epoch = []
    ev_per_neuron_per_epoch = []
    n_valid_per_epoch = []
    n_high_signal_per_epoch = []
    ct = 0

    for epoch in range(args.epochs):

        print("Start epoch", epoch)
        model.train()
        epoch_train_loss = 0

        for (image, behav, spikes) in train_dataloader:
            image, behav, spikes = image.to(device), behav.to(device), spikes.to(device)
            image = torch.squeeze(image, axis=1)
            pred = model(image, behav)
            loss = nn.functional.poisson_nll_loss(pred, spikes, reduction='mean', log_input=False)
            epoch_train_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        epoch_train_loss = epoch_train_loss / len(train_dataloader)
        train_loss_list.append(epoch_train_loss)

        if epoch_train_loss < best_train_loss:
            torch.save(model.state_dict(), args.best_train_path)
            best_train_loss = epoch_train_loss
            if len(val_dataloader) == 0:
                torch.save(model.state_dict(), args.best_val_path)

        print("Epoch {} train loss: {}".format(epoch, epoch_train_loss))

        # --- Validation ---
        model.eval()
        epoch_val_loss = 0
        pred_val_all = []
        label_val_all = []

        if len(val_dataloader) > 0:
            with torch.no_grad():
                for (image, behav, spikes) in val_dataloader:
                    image, behav, spikes = image.to(device), behav.to(device), spikes.to(device)
                    image = torch.squeeze(image, axis=1)
                    pred = model(image, behav)
                    loss = nn.functional.poisson_nll_loss(pred, spikes, reduction='mean', log_input=False)
                    epoch_val_loss += loss.item()
                    pred_val_all.append(pred.cpu().numpy())
                    label_val_all.append(spikes.cpu().numpy())

            epoch_val_loss = epoch_val_loss / len(val_dataloader)
            pred_val = np.concatenate(pred_val_all, axis=0)
            label_val = np.concatenate(label_val_all, axis=0)
            num_neurons = pred_val.shape[1]

            valid, high_signal = _compute_neuron_masks(pred_val, label_val)
            n_valid = int(valid.sum())
            n_hs = int(high_signal.sum())

            m_all = _metrics_for_mask(pred_val, label_val, valid, num_neurons)
            m_hs = _metrics_for_mask(pred_val, label_val, high_signal, num_neurons)

            val_cor_list.append(m_all["mean_cor"])
            val_r2_list.append(m_all["mean_r2"])
            val_mse_list.append(m_all["mse"])
            val_explained_var_list.append(m_all["mean_ev"])
            val_poisson_loss_list.append(float(epoch_val_loss))
            val_bits_per_spike_list.append(float(epoch_val_loss / np.log(2)))

            hs_cor_list.append(m_hs["mean_cor"])
            hs_r2_list.append(m_hs["mean_r2"])
            hs_mse_list.append(m_hs["mse"])
            hs_explained_var_list.append(m_hs["mean_ev"])

            cor_per_neuron_per_epoch.append(m_all["cor_pn"].copy())
            r2_per_neuron_per_epoch.append(m_all["r2_pn"].copy())
            ev_per_neuron_per_epoch.append(m_all["ev_pn"].copy())
            n_valid_per_epoch.append(n_valid)
            n_high_signal_per_epoch.append(n_hs)
        else:
            epoch_val_loss = np.inf
            nan = float("nan")
            for lst in [val_cor_list, val_r2_list, val_mse_list,
                        val_poisson_loss_list, val_bits_per_spike_list, val_explained_var_list,
                        hs_cor_list, hs_r2_list, hs_mse_list, hs_explained_var_list]:
                lst.append(nan)
            cor_per_neuron_per_epoch.append(None)
            r2_per_neuron_per_epoch.append(None)
            ev_per_neuron_per_epoch.append(None)
            n_valid_per_epoch.append(None)
            n_high_signal_per_epoch.append(None)

        val_loss_list.append(epoch_val_loss)

        if epoch_val_loss < best_val_loss:
            torch.save(model.state_dict(), args.best_val_path)
            best_val_loss = epoch_val_loss
            ct = 0
        else:
            ct += 1
            if len(val_dataloader) > 0 and ct > 5:
                print('stop training')
                break

        if len(val_dataloader) > 0:
            print(f"Epoch {epoch} val loss: {epoch_val_loss:.4f}")
            print(f"  ALL VALID ({n_valid}/{num_neurons}): corr={m_all['mean_cor']:.4f} R2={m_all['mean_r2']:.4f} MSE={m_all['mse']:.4f} EV={m_all['mean_ev']:.4f}")
            print(f"  HIGH SIG  ({n_hs}/{num_neurons}):  corr={m_hs['mean_cor']:.4f} R2={m_hs['mean_r2']:.4f} MSE={m_hs['mse']:.4f} EV={m_hs['mean_ev']:.4f}")
        else:
            print("Epoch {} val loss: {}".format(epoch, epoch_val_loss))

        print("End epoch", epoch)

    return {
        "train_loss_list": train_loss_list,
        "val_loss_list": val_loss_list,
        "val_cor_list": val_cor_list,
        "val_r2_list": val_r2_list,
        "val_mse_list": val_mse_list,
        "val_poisson_loss_list": val_poisson_loss_list,
        "val_bits_per_spike_list": val_bits_per_spike_list,
        "val_explained_var_list": val_explained_var_list,
        "hs_cor_list": hs_cor_list,
        "hs_r2_list": hs_r2_list,
        "hs_mse_list": hs_mse_list,
        "hs_explained_var_list": hs_explained_var_list,
        "cor_per_neuron_per_epoch": cor_per_neuron_per_epoch,
        "r2_per_neuron_per_epoch": r2_per_neuron_per_epoch,
        "ev_per_neuron_per_epoch": ev_per_neuron_per_epoch,
        "n_valid_per_epoch": n_valid_per_epoch,
        "n_high_signal_per_epoch": n_high_signal_per_epoch,
    }

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build datasets once to get num_neurons (use mean mode for quick probe)
_probe_ds = SensoriumDataset(root_dir=args.sensorium_root, data_split="train", seq_len=1,
                             vid_frame="mean", standardize_responses=True)
args.num_neurons = _probe_ds.num_neurons
del _probe_ds
print("num_neurons:", args.num_neurons)

# Construct a descriptive tag for filenames
if args.split_strategy == "tiers":
    _split_tag = "tiers"
else:
    _split_tag = f"random{int(args.train_ratio * 100)}"
if args.max_train_samples is not None:
    _split_tag += f"_max{args.max_train_samples}"
_frame_tag = "perframe" if args.vid_frame == "per_frame" else "mean"
_file_tag = f"{_split_tag}_{_frame_tag}"
print(f"file tag: {_file_tag}")

for shifter in [False, True]:
    print("\n====== shifter:", shifter, "======")
    model = Predictor(num_neurons=args.num_neurons).to(device)
    args.shifter = shifter
    args.best_train_path = "/home/herbelinluke/Downloads/paths/sensorium_trainCNNshifter_{}.pth".format(shifter)
    args.best_val_path = "/home/herbelinluke/Downloads/paths/sensorium_valCNNshifter_{}.pth".format(shifter)

    results = train_model()

    # --- Save results ---
    base_meta = {
        "model_name": "cnn",
        "file_id": "sensorium",
        "vid_type": "sensorium",
        "shifter": shifter,
        "dataset": "sensorium",
        "split_strategy": args.split_strategy,
        "train_ratio": args.train_ratio,
        "max_train_samples": args.max_train_samples,
        "vid_frame": args.vid_frame,
        "min_neuron_std": args.min_neuron_std,
        "train_loss_list": results["train_loss_list"],
        "val_loss_list": results["val_loss_list"],
    }
    fname = f"epoch_vs_score_cnn_sensorium_{_file_tag}_shifter_{shifter}.pkl"

    # Metric dirs: save both "all valid" and "high signal" variants
    score_dirs = [
        ("epoch_vs_score_data",              "val_cor_list",              results["val_cor_list"]),
        ("epoch_vs_correlation_data",        "val_cor_list",              results["val_cor_list"]),
        ("epoch_vs_r2_data",                 "val_r2_list",              results["val_r2_list"]),
        ("epoch_vs_mse_data",                "val_mse_list",             results["val_mse_list"]),
        ("epoch_vs_poisson_loss_data",       "val_poisson_loss_list",    results["val_poisson_loss_list"]),
        ("epoch_vs_bits_per_spike_data",     "val_bits_per_spike_list",  results["val_bits_per_spike_list"]),
        ("epoch_vs_explained_variance_data", "val_explained_var_list",   results["val_explained_var_list"]),
    ]
    for dir_name, score_key, score_list in score_dirs:
        os.makedirs(dir_name, exist_ok=True)
        save_path = os.path.join(dir_name, fname)
        with open(save_path, "wb") as f:
            pickle.dump({
                **base_meta, score_key: score_list,
                # high-signal variants alongside
                "hs_cor_list": results["hs_cor_list"],
                "hs_r2_list": results["hs_r2_list"],
                "hs_mse_list": results["hs_mse_list"],
                "hs_explained_var_list": results["hs_explained_var_list"],
                "n_valid_per_epoch": results["n_valid_per_epoch"],
                "n_high_signal_per_epoch": results["n_high_signal_per_epoch"],
            }, f)
        print("Saved to", save_path)

    per_neuron_dir = "epoch_vs_per_neuron_data"
    os.makedirs(per_neuron_dir, exist_ok=True)
    per_neuron_fname = f"per_neuron_cnn_sensorium_{_file_tag}_shifter_{shifter}.pkl"
    per_neuron_path = os.path.join(per_neuron_dir, per_neuron_fname)
    with open(per_neuron_path, "wb") as f:
        pickle.dump({
            **base_meta,
            "cor_per_neuron_per_epoch": results["cor_per_neuron_per_epoch"],
            "r2_per_neuron_per_epoch": results["r2_per_neuron_per_epoch"],
            "ev_per_neuron_per_epoch": results["ev_per_neuron_per_epoch"],
            "n_valid_per_epoch": results["n_valid_per_epoch"],
            "n_high_signal_per_epoch": results["n_high_signal_per_epoch"],
        }, f)
    print("Saved to", per_neuron_path)

num_neurons: 7863
file tag: random70_perframe

====== shifter: False ======
Loaded split from /home/herbelinluke/Downloads/dynamic29515-10-12-Video-9b4f6a1a067fe51e15306b9628efea20/meta/trials/split_70_30_all.pkl
split_strategy=random | train=17917 val=7679 | num_neurons=7863
Start epoch 0


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimod

Epoch 0 train loss: 0.5784187027386256


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimod

Epoch 0 val loss: 0.4467
  ALL VALID (7863/7863): corr=0.1031 R2=-0.0899 MSE=0.4758 EV=0.0060
  HIGH SIG  (7863/7863):  corr=0.1031 R2=-0.0899 MSE=0.4758 EV=0.0060
End epoch 0
Start epoch 1


...

Epoch 98 val loss: 0.2905
  ALL VALID (7863/7863): corr=0.2483 R2=0.0632 MSE=0.4115 EV=0.0635
  HIGH SIG  (7863/7863):  corr=0.2483 R2=0.0632 MSE=0.4115 EV=0.0635
End epoch 98
Start epoch 99


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimod

Epoch 99 train loss: 0.28571557189737046


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimod

Epoch 99 val loss: 0.2892
  ALL VALID (7863/7863): corr=0.2514 R2=0.0646 MSE=0.4109 EV=0.0649
  HIGH SIG  (7863/7863):  corr=0.2514 R2=0.0646 MSE=0.4109 EV=0.0649
End epoch 99
Saved to epoch_vs_score_data/epoch_vs_score_cnn_sensorium_random70_perframe_shifter_True.pkl
Saved to epoch_vs_correlation_data/epoch_vs_score_cnn_sensorium_random70_perframe_shifter_True.pkl
Saved to epoch_vs_r2_data/epoch_vs_score_cnn_sensorium_random70_perframe_shifter_True.pkl
Saved to epoch_vs_mse_data/epoch_vs_score_cnn_sensorium_random70_perframe_shifter_True.pkl
Saved to epoch_vs_poisson_loss_data/epoch_vs_score_cnn_sensorium_random70_perframe_shifter_True.pkl
Saved to epoch_vs_bits_per_spike_data/epoch_vs_score_cnn_sensorium_random70_perframe_shifter_True.pkl
Saved to epoch_vs_explained_variance_data/epoch_vs_score_cnn_sensorium_random70_perframe_shifter_True.pkl
Saved to epoch_vs_per_neuron_data/per_neuron_cnn_sensorium_random70_perframe_shifter_True.pkl


Evaluation

In [10]:
# Optional: smoothing (Sensorium is one sample per trial; small or no smoothing)
def smoothing_with_np_conv(nsp, size=5):
    if nsp.shape[0] < size:
        return nsp
    np_conv_res = []
    for i in range(nsp.shape[1]):
        np_conv_res.append(np.convolve(nsp[:, i], np.ones(size)/size, mode="same"))        
    np_conv_res = np.transpose(np.array(np_conv_res))
    return np_conv_res

In [11]:
def evaluate_model(model, weights_path, dataset, device):
    dl = DataLoader(dataset=dataset, batch_size=256, shuffle=False, num_workers=4)
    model.load_state_dict(torch.load(weights_path))
    ground_truth_all = []
    pred_all = []
    model.eval()
    with torch.no_grad():      
        for (image, behav, spikes) in dl:
            image = image.to(device)
            behav = behav.to(device)
            image = torch.squeeze(image, axis=1)
            pred = model(image, behav)
            ground_truth_all.append(spikes.numpy())
            pred_all.append(pred.cpu().numpy())
    return np.concatenate(pred_all, axis=0), np.concatenate(ground_truth_all, axis=0)

In [12]:
from sklearn.metrics import r2_score, mean_squared_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_ds = load_test_ds()

if len(test_ds) == 0:
    print("No test samples (len(test_ds)=0). Skipping evaluation.")
else:
    for shifter in [False, True]:
        print("\n====== shifter:", shifter, "======")
        args.shifter = shifter
        args.best_val_path = "/home/herbelinluke/Downloads/paths/sensorium_valCNNshifter_{}.pth".format(shifter)
        model = Predictor(num_neurons=args.num_neurons).to(device)
        pred, label = evaluate_model(model, weights_path=args.best_val_path, dataset=test_ds, device=device)

        num_neurons = pred.shape[1]
        valid, high_signal = _compute_neuron_masks(pred, label)

        for mask_name, mask in [("ALL VALID", valid), ("HIGH SIGNAL", high_signal)]:
            n = int(mask.sum())
            m = _metrics_for_mask(pred, label, mask, num_neurons)
            print(f"\n  {mask_name} ({n}/{num_neurons} neurons):")
            print(f"    corr  = {m['mean_cor']:.4f} +/- {np.nanstd(m['cor_pn']):.4f}  "
                  f"[{np.nanmin(m['cor_pn']):.4f}, {np.nanmax(m['cor_pn']):.4f}]")
            print(f"    R2    = {m['mean_r2']:.4f}")
            print(f"    MSE   = {m['mse']:.6f}")
            print(f"    EV    = {m['mean_ev']:.4f}")

Loaded split from /home/herbelinluke/Downloads/dynamic29515-10-12-Video-9b4f6a1a067fe51e15306b9628efea20/meta/trials/split_70_30_all.pkl
Test set: random val holdout (7679 trials)

====== shifter: False ======


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:59: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(resp[start:end], axis=0)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse


  ALL VALID (7863/7863 neurons):
    corr  = 0.2545 +/- 0.0568  [0.0948, 0.5260]
    R2    = 0.0660
    MSE   = 0.410088
    EV    = 0.0666

  HIGH SIGNAL (7863/7863 neurons):
    corr  = 0.2545 +/- 0.0568  [0.0948, 0.5260]
    R2    = 0.0660
    MSE   = 0.410088
    EV    = 0.0666

====== shifter: True ======


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:59: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(resp[start:end], axis=0)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse


  ALL VALID (7863/7863 neurons):
    corr  = 0.2514 +/- 0.0559  [0.1029, 0.5119]
    R2    = 0.0646
    MSE   = 0.410939
    EV    = 0.0649

  HIGH SIGNAL (7863/7863 neurons):
    corr  = 0.2514 +/- 0.0559  [0.1029, 0.5119]
    R2    = 0.0646
    MSE   = 0.410939
    EV    = 0.0649
